# Battle 通用远程 GPU Worker（单 cell 极简版）

**用法**
- **Push（推荐，无本地 hub）**：`mode="push"`，只填 `push_token`。Run → 打印隧道 URL → 控制台 Push 启动 trainer → hub 推 `code.zip`+job。
- **Pull**：`mode="pull"` + `hub_url`/`hub_token`。

| 步骤 | 住哪 |
|---|---|
| CFG / 保活 / 薄分支 | 本 cell |
| Push 引导（升级 HTTP + 隧道，~70 行） | 本 cell（**首包前无 code.zip**，只能内联） |
| 设备探测 / 守候日志 / 监督 / cloudflared 完整逻辑 | `remote/notebook_runtime.py` + `remote/push_bootstrap.py`（code.zip） |

修运行时 → 改仓库后重跑 cell / 换 job 即可，**不必重发 notebook**（除改 CFG 或引导骨架）。

**停止**：■ 中断本 cell。Kaggle 单次 9h。


In [ ]:
# @title Battle GPU Worker —— Push 无 hub；Pull 填 hub_url
# 引导骨架（参数/保活/薄分支）留在本 cell；其余逻辑在 code.zip
# （remote/notebook_runtime.py + remote/push_bootstrap.py）。
import os, sys, threading, time

CFG = {
    "mode": "push",              # push | pull
    "hub_url": "",               # pull 必填
    "hub_token": "",
    "push_port": 8790,
    "push_token": "YOUR_TOKEN_HERE",
    "cloudflared_path": "",
    "device": "auto",
    "use_multi_gpu": True,
    "max_session_hours": 9,
    "poll_interval_sec": 5,
    "idle_floor_sec": 3600,
    "max_worker_restarts": 5,
}

def _log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] {msg}", flush=True)

_keepalive_stop = threading.Event()

def _keepalive_loop():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
        while not _keepalive_stop.is_set():
            try:
                from IPython.display import Javascript, display
                display(Javascript(
                    "function f(){document.querySelector('colab-connect-button')?.click();}"
                    "setTimeout(f,1000);"))
            except Exception:
                pass
            _keepalive_stop.wait(60)
    else:
        n = 0
        while not _keepalive_stop.is_set():
            print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] [keepalive] alive ({n*2} min)", flush=True)
            n += 1
            _keepalive_stop.wait(120)

threading.Thread(target=_keepalive_loop, daemon=True, name="keepalive").start()
_log("Keepalive 已启动")

_hub = str(CFG.get("hub_url") or "").strip()
_mode = str(CFG.get("mode") or "pull").strip().lower()
_code_dir = "/tmp/worker-code"

# ── Pull / push-with-hub：GET /code → code.zip 运行时 ─────────────────
if _mode == "pull" or (_mode == "push" and _hub and "your-tunnel" not in _hub):
    if not _hub:
        raise SystemExit("[battle-rl] FATAL: pull 需要 hub_url")
    import hashlib, io, urllib.error, urllib.request, zipfile
    from pathlib import Path
    _deadline = time.time() + 3600
    _attempt = 0
    while True:
        _attempt += 1
        try:
            _req = urllib.request.Request(
                _hub.rstrip("/") + "/code",
                headers={"Authorization": "Bearer " + str(CFG["hub_token"])})
            with urllib.request.urlopen(_req, timeout=120) as _resp:
                _raw = _resp.read()
            _log(f"code.zip 就绪: {len(_raw)} bytes sha={hashlib.sha256(_raw).hexdigest()[:12]}…")
            break
        except urllib.error.HTTPError as _e:
            if _e.code in (401, 403):
                raise SystemExit(f"[battle-rl] FATAL: /code HTTP {_e.code} — token 不一致") from None
            if _e.code != 404:
                _log(f"/code HTTP {_e.code} —— 30s 后重试")
            elif time.time() > _deadline:
                raise SystemExit("[battle-rl] FATAL: 等 code.zip 超 1h") from None
            time.sleep(30)
        except Exception as _e:
            _log(f"hub 连接异常（{type(_e).__name__}）—— 30s 后重试")
            time.sleep(30)
    Path(_code_dir).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(_raw)) as _z:
        _z.extractall(_code_dir)
    sys.path.insert(0, _code_dir)
    _log("移交 remote.notebook_runtime")
    from remote.notebook_runtime import run_notebook
    CFG["keepalive_stop"] = _keepalive_stop
    CFG["log"] = _log
    CFG["code_dir"] = _code_dir
    raise SystemExit(run_notebook(CFG))

# ── Push-first：无 hub /code。升级 HTTP + 隧道留在本 cell（首包前无 code）──
if CFG.get("push_token", "") in ("", "YOUR_TOKEN_HERE"):
    raise SystemExit("[battle-rl] FATAL: push_token 必填")
import base64, io, json, secrets, subprocess, zipfile
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.request import Request, urlopen

_port = int(CFG["push_port"])
_tok = str(CFG["push_token"])
_work = Path("/tmp/remote-worker-serve"); _work.mkdir(parents=True, exist_ok=True)
_cdir = Path(_code_dir)
_upg = {"body": None}

class _H(BaseHTTPRequestHandler):
    def log_message(self, *a): pass
    def _j(self, o, s=200):
        b = json.dumps(o).encode(); self.send_response(s)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(b))); self.end_headers(); self.wfile.write(b)
    def _ok(self):
        return secrets.compare_digest(self.headers.get("Authorization", ""), f"Bearer {_tok}")
    def do_GET(self):
        if not self._ok(): return self._j({"error": "unauthorized"}, 401)
        p = self.path.split("?", 1)[0]
        if p == "/ping":
            return self._j({"ok": True, "busy": False, "queued": 0, "done": 0, "bootstrap": True})
        if p == "/code-sha": return self._j({"cached": False})
        return self._j({"error": "nf"}, 404)
    def do_POST(self):
        if not self._ok(): return self._j({"error": "unauthorized"}, 401)
        if self.path.split("?", 1)[0] != "/job": return self._j({"error": "nf"}, 404)
        raw = self.rfile.read(int(self.headers.get("Content-Length", "0")))
        try: body = json.loads(raw.decode())
        except ValueError: return self._j({"error": "bad json"}, 400)
        if not body.get("code_b64"):
            return self._j({"error": "code-missing"}, 428)
        try:
            _cdir.mkdir(parents=True, exist_ok=True)
            zipfile.ZipFile(io.BytesIO(base64.b64decode(body["code_b64"]))).extractall(_cdir)
        except Exception as e:
            return self._j({"error": str(e)}, 400)
        _upg["body"] = raw
        _log(f"code.zip 已解包 -> {_cdir}（升级完整 worker_server）")
        return self._j({"status": "accepted", "upgrading": True}, 202)

_srv = ThreadingHTTPServer(("0.0.0.0", _port), _H)
threading.Thread(target=_srv.serve_forever, daemon=True).start()
_log(f"push bootstrap :{_port}（无 hub /code）")

_cf = str(CFG.get("cloudflared_path") or "") or os.popen(
    "where cloudflared 2>nul || which cloudflared 2>/dev/null").read().strip()
if not _cf:
    _log("cloudflared 安装…")
    _cf = "/usr/local/bin/cloudflared"
    subprocess.run(["curl", "-fsSL",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-o", _cf], check=True, timeout=60)
    os.chmod(_cf, 0o755)
_cf_log = _work / "cloudflared.log"
_cfp = subprocess.Popen(
    [_cf, "tunnel", "--url", f"http://localhost:{_port}", "--logfile", str(_cf_log)],
    stdout=open(_cf_log, "w"), stderr=subprocess.STDOUT)
_url, _t0 = None, time.time()
while time.time() - _t0 < 60 and not _url:
    try:
        import re as _re
        _urls = _re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com",
                            Path(_cf_log).read_text(encoding="utf-8", errors="replace"))
        if _urls: _url = _urls[-1]; break
    except OSError:
        pass
    if _cfp.poll() is not None: break
    time.sleep(2)
if _url:
    _log(f"★ 隧道 URL: {_url}")
    _log("填到控制台 Push（auth key=push_token）或 rl-config gpu_push；等首个 job…")
else:
    _log(f"⚠ 无隧道 URL——查 {_cf_log}")

_deadline = time.time() + int(CFG["max_session_hours"]) * 3600
try:
    while _upg["body"] is None:
        if time.time() > _deadline:
            raise SystemExit(0)
        time.sleep(2)
except KeyboardInterrupt:
    _log("收到中断"); raise SystemExit(0)
finally:
    _srv.shutdown(); time.sleep(0.5)

# code 已在 _cdir —— 之后全部走 code.zip 运行时
sys.path.insert(0, _code_dir)
from remote.push_bootstrap import requeue_job, spawn_full_worker_server, wait_ping
from remote.notebook_runtime import run_notebook

_real = spawn_full_worker_server(
    _port, _tok, _work, str(CFG.get("device") or "cpu"), _cdir, _work / "serve.log")
if not wait_ping(_port, _tok, 30):
    _log("worker_server 30s 未就绪"); _real.kill(); raise SystemExit(-1)
_log("worker_server 就绪——重放首个 job 后守候")
requeue_job(_port, _tok, _upg["body"] or b"{}")
CFG["keepalive_stop"] = _keepalive_stop
CFG["log"] = _log
CFG["code_dir"] = _code_dir
CFG["already_serving"] = {
    "serve_pid": _real.pid,
    "cf_pid": _cfp.pid if _cfp else None,
    "cf_url": _url,
}
raise SystemExit(run_notebook(CFG))
